# ASSIGNMENT 1 - Question 3 - Crack Detection from drone Imagery
### Alumno: Martín Andrés Gutiérrez López
Topics Sobel, Laplacian Spatian filters, receptive fields

In [2]:
# ==============================================================================
# Question 3: Crack Detection from Drone Imagery (RoadScan)
# Solución de filtros espaciales (Sobel/Laplaciano) y cálculo de Receptive Field
# ==============================================================================

import numpy as np

BOLD = "\033[1m"
BOLD_BLUE = "\033[1;34m"
RESET = "\033[0m"

# ------------------------------------------------------------------------------
# Incisos (a) y (b): Operaciones con filtros espaciales en el parche 3x3
# ------------------------------------------------------------------------------
# Parche de imagen
I = np.array([[10, 10, 80], [10, 10, 80], [10, 10, 80]], dtype=float)

# Kernels
K_sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)

K_sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=float)

K_laplacian = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=float)

# Cálculo de convoluciones en el píxel central (producto Hadamard y suma)
Gx = np.sum(I * K_sobel_x)
Gy = np.sum(I * K_sobel_y)
grad_mag = np.sqrt(Gx**2 + Gy**2)
grad_dir_rad = np.arctan2(Gy, Gx)
grad_dir_deg = np.degrees(grad_dir_rad)
lap_response = np.sum(I * K_laplacian)

print(f"\n{BOLD_BLUE}")
print("=" * 70)
print("RESPUESTAS INCISOS (a) Y (b): FILTROS ESPACIALES")
print("=" * 70,f"{RESET}")

print(f"Gx (Sobel horizontal)      : {Gx:.1f}")
print(f"Gy (Sobel vertical)        : {Gy:.1f}")
print(f"Magnitud del Gradiente     : {grad_mag:.1f}")
print(f"Dirección del Gradiente    : {grad_dir_deg:.1f}° (0 rad)")
print("Interpretación de dirección : Vector hacia la derecha; borde vertical.")
print(f"Respuesta del Laplaciano   : {lap_response:.1f}")
print()

# ------------------------------------------------------------------------------
# Inciso (c): Cálculo del Campo Receptivo (Receptive Field) paso a paso
# ------------------------------------------------------------------------------
layers = [
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1},
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1},
    {"name": "MaxPool 2x2 (s=2)", "k": 2, "s": 2},
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1},
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1},
]

rf = 1  # Receptive field inicial en la entrada
j = 1  # Salto acumulado inicial

print(f"\n{BOLD_BLUE}")
print("=" * 70)
print("RESPUESTA INCISO (c): TABLA DE CAMPO RECEPTIVO")
print("=" * 70,f"{RESET}")
print(f"{'Capa':<20} | {'Kernel (k)':<10} | {'Stride (s)':<10} | {'Jump (j)':<10} | {'RF':<6}")
print("-" * 70)
print(f"{'Input':<20} | {'-':<10} | {'-':<10} | {j:<10} | {rf:<6}")

for layer in layers:
  k = layer["k"]
  s = layer["s"]

  # Fórmula: RF_out = RF_in + (k - 1) * j_in
  rf = rf + (k - 1) * j
  # Actualización del salto: j_out = j_in * s
  j = j * s

  print(
      f"{layer['name']:<20} | {k:<10} | {s:<10} | {j:<10} | {rf:<6}"
  )

print("-" * 70)
target_crack_size = 20
is_sufficient = rf >= target_crack_size
print(f"Campo receptivo final alcanzado : {rf} píxeles")
print(f"¿Suficiente para grietas de {target_crack_size} px?  : {'SÍ' if is_sufficient else 'NO'}")



RESPUESTAS INCISOS (a) Y (b): FILTROS ESPACIALES
Gx (Sobel horizontal)      : 280.0
Gy (Sobel vertical)        : 0.0
Magnitud del Gradiente     : 280.0
Dirección del Gradiente    : 0.0° (0 rad)
Interpretación de dirección : Vector hacia la derecha; borde vertical.
Respuesta del Laplaciano   : 70.0



RESPUESTA INCISO (c): TABLA DE CAMPO RECEPTIVO
Capa                 | Kernel (k) | Stride (s) | Jump (j)   | RF    
----------------------------------------------------------------------
Input                | -          | -          | 1          | 1     
Conv 3x3 (s=1)       | 3          | 1          | 1          | 3     
Conv 3x3 (s=1)       | 3          | 1          | 1          | 5     
MaxPool 2x2 (s=2)    | 2          | 2          | 2          | 6     
Conv 3x3 (s=1)       | 3          | 1          | 2          | 10    
Conv 3x3 (s=1)       | 3          | 1          | 2          | 14    
----------------------------------------------------------------------
Campo receptivo final al

### Conclusion: 
The final receptive field is only 14 pixels, so a neuron in the last layer cannot encompass the entire structure of a 20-pixel-wide crack (spatial context is missing).
Two proposed changes:
- Add a second 2x2 MaxPool layer (s=2) or increase the stride: For example, placing an additional MaxPool layer before Layer 4 or using a stride of 2 in one of the convolutions would increase the step to 4, doubling the RF growth rate.
- Use Dilated Convolutions (Atrous / Dilated Convolutions): Introducing dilation (e.g., d=2 or d=3) in layers 4 and 5 drastically expands the receptive field without increasing the number of trainable parameters or losing spatial resolution.

The Next Code **resolve this situation for both methods**.

In [9]:
# ==============================================================================
# EXTENSION: Implementation of the Two Proposed Changes (Target: RF >= 20 pixels)
# ==============================================================================


def compute_rf_pipeline(pipeline_name, layers, target=20):
  rf = 1  # Initial receptive field at input
  j = 1  # Initial cumulative jump (stride product)
  
  print(f"\n{BOLD_BLUE}")
  print("=" * 86)
  print(f"PIPELINE: {pipeline_name}")
  print("=" * 86,f"{RESET}")
  print(f"{'Layer':<28} | {'Kernel (k)':<10} | {'Stride (s)':<10} | {'Dilation (d)':<12} | {'Jump (j)':<9} | {'RF':<6}")
  print("-" * 86)
  print(
      f"{'Input':<28} | {'-':<10} | {'-':<10} | {'-':<12} | {j:<9} | {rf:<6}"
  )

  for layer in layers:
    k = layer["k"]
    s = layer["s"]
    d = layer.get("d", 1)  # Dilation factor (defaults to 1)

    # Effective kernel size with dilation: k_eff = d * (k - 1) + 1
    k_eff = d * (k - 1) + 1

    # Receptive field and jump update formulas
    rf = rf + (k_eff - 1) * j
    j = j * s

    print(
        f"{layer['name']:<28} | {k:<10} | {s:<10} | {d:<12} | {j:<9} | {rf:<6}"
    )

  print("-" * 86)
  meets_target = rf >= target
  print(
      f"Final Receptive Field: {rf} pixels (Target: {target} px) ->"
      f" {'MET (>= 20)' if meets_target else 'INSUFFICIENT'}\n"
  )
  return rf


# ------------------------------------------------------------------------------
# PROPOSAL 1: Add a second MaxPool 2x2 (s=2) layer before Layer 5
# This doubles the stride jump to j=4, scaling the last Conv layer expansion
# ------------------------------------------------------------------------------
pipeline_proposal_1 = [
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1, "d": 1},
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1, "d": 1},
    {"name": "MaxPool 2x2 (s=2)", "k": 2, "s": 2, "d": 1},
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1, "d": 1},
    {"name": "MaxPool 2x2 (s=2) [NEW]", "k": 2, "s": 2, "d": 1},
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1, "d": 1},
]

rf_1 = compute_rf_pipeline(
    "Proposal 1: Additional MaxPool 2x2 (s=2)", pipeline_proposal_1
)


# ------------------------------------------------------------------------------
# PROPOSAL 2: Dilated Convolutions (Atrous Convolutions with d=2 and d=3)
# Expands receptive field without spatial downsampling or parameter overhead
# ------------------------------------------------------------------------------
pipeline_proposal_2 = [
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1, "d": 1},
    {"name": "Conv 3x3 (s=1)", "k": 3, "s": 1, "d": 1},
    {"name": "MaxPool 2x2 (s=2)", "k": 2, "s": 2, "d": 1},
    {"name": "Dilated Conv 3x3 (d=2)", "k": 3, "s": 1, "d": 2},
    {"name": "Dilated Conv 3x3 (d=3)", "k": 3, "s": 1, "d": 3},
]

rf_2 = compute_rf_pipeline(
    "Proposal 2: Dilated Convolutions (d=2, d=3)", pipeline_proposal_2
)



PIPELINE: Proposal 1: Additional MaxPool 2x2 (s=2)
Layer                        | Kernel (k) | Stride (s) | Dilation (d) | Jump (j)  | RF    
--------------------------------------------------------------------------------------
Input                        | -          | -          | -            | 1         | 1     
Conv 3x3 (s=1)               | 3          | 1          | 1            | 1         | 3     
Conv 3x3 (s=1)               | 3          | 1          | 1            | 1         | 5     
MaxPool 2x2 (s=2)            | 2          | 2          | 1            | 2         | 6     
Conv 3x3 (s=1)               | 3          | 1          | 1            | 2         | 10    
MaxPool 2x2 (s=2) [NEW]      | 2          | 2          | 1            | 4         | 12    
Conv 3x3 (s=1)               | 3          | 1          | 1            | 4         | 20    
--------------------------------------------------------------------------------------
Final Receptive Field: 20 pixels (Target: 20 